In [1]:
!pip install openai requests -q
print("설치 완료!")

설치 완료!


In [2]:
import urllib.request
import gzip
import json

# Download first few lines to confirm we can read the file
url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Gift_Cards.jsonl.gz"

print("Downloading a sample of the dataset...")
with urllib.request.urlopen(url) as response:
    with gzip.GzipFile(fileobj=response) as f:
        reviews = []
        for i, line in enumerate(f):
            reviews.append(json.loads(line))
            if i >= 4:  # just grab 5 lines to confirm it works
                break

print(f"Successfully read {len(reviews)} reviews.")
print("\nSample review fields:")
print(json.dumps(reviews[0], indent=2))

Successfully read 5 reviews.

Sample review fields:
{
  "rating": 5.0,
  "title": "Great gift",
  "text": "Having Amazon money is always good.",
  "images": [],
  "asin": "B00IX1I3G6",
  "parent_asin": "B00IX1I3G6",
  "user_id": "AHZ6XMOLEWA67S3TX7IWEXXGWSOA",
  "timestamp": 1549866158332,
  "helpful_vote": 0,
  "verified_purchase": true
}


In [3]:
from openai import OpenAI

# Set up the client with the professor's endpoint
client = OpenAI(
    base_url="http://dobolyi.com:9001/v1",
    api_key="6418"
)

def classify_sentiment(title: str, text: str) -> str:
    """
    Classifies a review as POSITIVE or NEGATIVE using an LLM.
    Never uses the star rating — only the title and text.
    """
    prompt = f"""You are a sentiment classifier for product reviews.

Classify the following review as exactly one word: POSITIVE or NEGATIVE.

Rules:
- Base your decision only on the title and text. Do not guess a star rating.
- If the title and text conflict, weight the text more heavily.
- Short, sarcastic, or angry reviews with no praise → NEGATIVE.
- Any genuine satisfaction, even minor, with no clear complaint → POSITIVE.
- Respond with ONLY the single word: POSITIVE or NEGATIVE. Nothing else.

Review Title: {title}
Review Text: {text}

Sentiment:"""

    response = client.chat.completions.create(
        model="cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=5,
        temperature=0
    )

    result = response.choices[0].message.content.strip().upper()
    # Clean up in case the model adds extra words
    if "POSITIVE" in result:
        return "POSITIVE"
    elif "NEGATIVE" in result:
        return "NEGATIVE"
    else:
        return "POSITIVE"  # default fallback

# Quick spot-check on 2 obvious cases
test_cases = [
    ("Amazing gift card!", "Absolutely love it. Fast delivery and worked perfectly. Will buy again!"),
    ("Waste of money", "This card never worked. Tried for weeks, customer service was useless. Total scam.")
]

print("=== Spot-check ===")
for title, text in test_cases:
    result = classify_sentiment(title, text)
    print(f"Title: {title}")
    print(f"Result: {result}\n")

=== Spot-check ===


AttributeError: 'NoneType' object has no attribute 'strip'

In [4]:
# Debug: see the raw response
response = client.chat.completions.create(
    model="cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit",
    messages=[{"role": "user", "content": "Reply with one word: POSITIVE"}],
    max_tokens=20,
    temperature=0
)
print(response)

ChatCompletion(id='chatcmpl-92f3351aa3b8ca0c', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="Here's a thinking process:\n\n1.  **Analyze User Input:**\n   - User"), stop_reason=None, token_ids=None, routed_experts=None)], created=1789997429, model='cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='vllm-0.29.0-544d88b9', usage=CompletionUsage(completion_tokens=20, prompt_tokens=17, total_tokens=37, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=20, rejected_prediction_tokens=None), prompt_tokens_details=None), prompt_logprobs=None, prompt_token_ids=None, prompt_text=None, kv_transfer_params=None, ec_transfer_params=None, metrics=None)


In [5]:
def classify_sentiment(title: str, text: str) -> str:
    """
    Classifies a review as POSITIVE or NEGATIVE using an LLM.
    Never uses the star rating — only the title and text.
    """
    prompt = f"""You are a sentiment classifier for product reviews. /no_think

Classify the following review as exactly one word: POSITIVE or NEGATIVE.

Rules:
- Base your decision only on the title and text. Do not guess a star rating.
- If the title and text conflict, weight the text more heavily.
- Short, sarcastic, or angry reviews with no praise → NEGATIVE.
- Any genuine satisfaction, even minor, with no clear complaint → POSITIVE.
- Respond with ONLY the single word: POSITIVE or NEGATIVE. Nothing else.

Review Title: {title}
Review Text: {text}

Sentiment:"""

    response = client.chat.completions.create(
        model="cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
        temperature=0
    )

    result = response.choices[0].message.content
    if result is None:
        # fallback: check reasoning field
        result = response.choices[0].message.reasoning or ""
    result = result.strip().upper()

    if "POSITIVE" in result:
        return "POSITIVE"
    elif "NEGATIVE" in result:
        return "NEGATIVE"
    else:
        return "POSITIVE"  # default fallback

# Quick spot-check
test_cases = [
    ("Amazing gift card!", "Absolutely love it. Fast delivery and worked perfectly. Will buy again!"),
    ("Waste of money", "This card never worked. Tried for weeks, customer service was useless. Total scam.")
]

print("=== Spot-check ===")
for title, text in test_cases:
    result = classify_sentiment(title, text)
    print(f"Title: {title}")
    print(f"Result: {result}\n")

=== Spot-check ===
Title: Amazing gift card!
Result: POSITIVE

Title: Waste of money
Result: POSITIVE



In [6]:
# Debug the negative case
response = client.chat.completions.create(
    model="cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit",
    messages=[{"role": "user", "content": """You are a sentiment classifier. /no_think

Classify this review as POSITIVE or NEGATIVE. Reply with ONE word only.

Review Title: Waste of money
Review Text: This card never worked. Tried for weeks, customer service was useless. Total scam.

Sentiment:"""}],
    max_tokens=50,
    temperature=0
)
print("content:", repr(response.choices[0].message.content))
print("reasoning:", repr(response.choices[0].message.reasoning))
print("finish_reason:", response.choices[0].finish_reason)


content: None
reasoning: "Here's a thinking process:\n\n1.  **Analyze User Input:**\n   - **Role:** Sentiment classifier\n   - **Constraint:** `/no_think` (implies I should not output any reasoning, just the final answer"
finish_reason: length


In [7]:
def classify_sentiment(title: str, text: str) -> str:
    """
    Classifies a review as POSITIVE or NEGATIVE using an LLM.
    Never uses the star rating — only the title and text.
    """
    prompt = f"""Classify the following Amazon review as exactly one word: POSITIVE or NEGATIVE.

Rules:
- Base your decision only on the title and text. Do not use any star rating.
- If the title and text conflict, weight the text more heavily.
- Short, sarcastic, or angry reviews with no praise → NEGATIVE.
- Any genuine satisfaction, even minor, with no clear complaint → POSITIVE.
- Reply with ONLY one word: POSITIVE or NEGATIVE.

Review Title: {title}
Review Text: {text}

Sentiment:"""

    response = client.chat.completions.create(
        model="cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
        temperature=0,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )

    result = response.choices[0].message.content
    if result is None:
        result = ""
    result = result.strip().upper()

    if "POSITIVE" in result:
        return "POSITIVE"
    elif "NEGATIVE" in result:
        return "NEGATIVE"
    else:
        return "POSITIVE"  # default fallback

# Quick spot-check
test_cases = [
    ("Amazing gift card!", "Absolutely love it. Fast delivery and worked perfectly. Will buy again!"),
    ("Waste of money", "This card never worked. Tried for weeks, customer service was useless. Total scam.")
]

print("=== Spot-check ===")
for title, text in test_cases:
    result = classify_sentiment(title, text)
    print(f"Title: {title}")
    print(f"Result: {result}\n")

=== Spot-check ===
Title: Amazing gift card!
Result: POSITIVE

Title: Waste of money
Result: NEGATIVE



In [8]:
import urllib.request
import gzip
import json
import time

# Step 2: Load first 100 reviews and score them
url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Gift_Cards.jsonl.gz"

print("Loading 100 reviews from dataset...")
raw_reviews = []
with urllib.request.urlopen(url) as response:
    with gzip.GzipFile(fileobj=response) as f:
        for i, line in enumerate(f):
            raw_reviews.append(json.loads(line))
            if i >= 99:
                break

print(f"Loaded {len(raw_reviews)} reviews.")

# Convert star rating to ground truth label (model never sees this)
def rating_to_label(rating):
    return "POSITIVE" if rating >= 4 else "NEGATIVE"

# Run classification on all 100 reviews
print("\nClassifying reviews (this may take a few minutes)...")
results = []
for i, review in enumerate(raw_reviews):
    title = review.get("title",

_IncompleteInputError: incomplete input (1641650083.py, line 28)

In [9]:
import urllib.request
import gzip
import json

url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Gift_Cards.jsonl.gz"

print("Loading 100 reviews from dataset...")
raw_reviews = []
with urllib.request.urlopen(url) as response:
    with gzip.GzipFile(fileobj=response) as f:
        for i, line in enumerate(f):
            raw_reviews.append(json.loads(line))
            if i >= 99:
                break

print(f"Loaded {len(raw_reviews)} reviews.")

def rating_to_label(rating):
    return "POSITIVE" if rating >= 4 else "NEGATIVE"

print("\nClassifying reviews (this may take a few minutes)...")
results = []
for i, review in enumerate(raw_reviews):
    title = review.get("title") or ""
    text = review.get("text") or ""
    rating = review["rating"]
    predicted = classify_sentiment(title, text)
    ground_truth = rating_to_label(rating)
    correct = (predicted == ground_truth)
    results.append({
        "index": i,
        "rating": rating,
        "ground_truth": ground_truth,
        "predicted": predicted,
        "correct": correct,
        "title": title,
        "text": text[:200]
    })
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/100...")

total = len(results)
correct_count = sum(1 for r in results if r["correct"])
accuracy = correct_count / total * 100

print(f"\n=== Results ===")
print(f"Overall accuracy: {correct_count}/{total} = {accuracy:.1f}%")

for label in ["POSITIVE", "NEGATIVE"]:
    class_rows = [r for r in results if r["ground_truth"] == label]
    class_correct = sum(1 for r in class_rows if r["correct"])
    if class_rows:
        pct = class_correct / len(class_rows) * 100
        print(f"{label}: {class_correct}/{len(class_rows)} correct ({pct:.1f}%)")

wrong = [r for r in results if not r["correct"]]
print(f"\nWrong predictions: {len(wrong)}")
for r in wrong[:5]:
    print(f"  [#{r['index']}] Rating={r['rating']} -> GT={r['ground_truth']}, Predicted={r['predicted']}")
    print(f"  Title: {r['title'][:60]}")

Loading 100 reviews from dataset...
Loaded 100 reviews.

Classifying reviews (this may take a few minutes)...
  Processed 10/100...
  Processed 20/100...
  Processed 30/100...
  Processed 40/100...
  Processed 50/100...
  Processed 60/100...
  Processed 70/100...
  Processed 80/100...
  Processed 90/100...
  Processed 100/100...

=== Results ===
Overall accuracy: 98/100 = 98.0%
POSITIVE: 92/93 correct (98.9%)
NEGATIVE: 6/7 correct (85.7%)

Wrong predictions: 2
  [#46] Rating=5.0 -> GT=POSITIVE, Predicted=NEGATIVE
  Title: Love it!!
  [#98] Rating=3.0 -> GT=NEGATIVE, Predicted=POSITIVE
  Title: Easy to use


In [10]:
import json

# Save results to JSON for use in dashboard later
with open("results_100.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved results_100.json")
print(f"Total: {total}, Correct: {correct_count}, Accuracy: {accuracy:.1f}%")
print(f"Class distribution: {sum(1 for r in results if r['ground_truth']=='POSITIVE')} POSITIVE, {sum(1 for r in results if r['ground_truth']=='NEGATIVE')} NEGATIVE")


Saved results_100.json
Total: 100, Correct: 98, Accuracy: 98.0%
Class distribution: 93 POSITIVE, 7 NEGATIVE


In [11]:
import json

with open("results_100.json") as f:
    results = json.load(f)

total = len(results)
correct_count = sum(1 for r in results if r["correct"])
accuracy = correct_count / total * 100
pos_rows = [r for r in results if r["ground_truth"] == "POSITIVE"]
neg_rows = [r for r in results if r["ground_truth"] == "NEGATIVE"]
pos_correct = sum(1 for r in pos_rows if r["correct"])
neg_correct = sum(1 for r in neg_rows if r["correct"])

results_json = json.dumps(results)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Amazon Review Sentiment Dashboard</title>
<style>
  :root {{
    --bg: #f8f9fc; --card: #ffffff; --border: #e2e8f0;
    --text: #1a202c; --muted: #718096;
    --pos: #38a169; --neg: #e53e3e; --neutral: #718096;
    --accent: #3b82f6; --accent-dark: #2563eb;
  }}
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Segoe UI', system-ui, sans-serif; background: var(--bg); color: var(--text); }}
  header {{ background: var(--accent-dark); color: white; padding: 28px 40px; }}
  header h1 {{ font-size: 1.6rem; font-weight: 700; }}
  header p {{ font-size: 0.9rem; opacity: 0.8; margin-top: 4px; }}
  .container {{ max-width: 1100px; margin: 32px auto; padding: 0 24px; }}
  .stats {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin-bottom: 28px; }}
  .stat-card {{ background: var(--card); border: 1px solid var(--border); border-radius: 10px; padding: 20px 24px; }}
  .stat-card .label {{ font-size: 0.78rem; color: var(--muted); text-transform: uppercase; letter-spacing: 0.05em; }}
  .stat-card .value {{ font-size: 2rem; font-weight: 700; margin-top: 6px; }}
  .stat-card .sub {{ font-size: 0.82rem; color: var(--muted); margin-top: 2px; }}
  .card {{ background: var(--card); border: 1px solid var(--border); border-radius: 10px; padding: 24px; margin-bottom: 24px; }}
  .card h2 {{ font-size: 1rem; font-weight: 600; margin-bottom: 16px; }}
  .bar-row {{ display: flex; align-items: center; gap: 12px; margin-bottom: 10px; }}
  .bar-label {{ width: 80px; font-size: 0.85rem; font-weight: 600; }}
  .bar-track {{ flex: 1; background: var(--bg); border-radius: 6px; height: 22px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 6px; display: flex; align-items: center; padding-left: 8px; font-size: 0.78rem; color: white; font-weight: 600; transition: width 0.6s ease; }}
  .bar-pct {{ font-size: 0.85rem; color: var(--muted); width: 55px; text-align: right; }}
  .filters {{ display: flex; gap: 10px; margin-bottom: 14px; flex-wrap: wrap; align-items: center; }}
  .filter-btn {{ padding: 6px 14px; border: 1px solid var(--border); border-radius: 20px; background: white; cursor: pointer; font-size: 0.83rem; transition: all 0.15s; }}
  .filter-btn.active {{ background: var(--accent); color: white; border-color: var(--accent); }}
  .count-label {{ margin-left: auto; font-size: 0.83rem; color: var(--muted); }}
  table {{ width: 100%; border-collapse: collapse; font-size: 0.85rem; }}
  th {{ text-align: left; padding: 10px 12px; border-bottom: 2px solid var(--border); color: var(--muted); font-weight: 600; font-size: 0.78rem; text-transform: uppercase; }}
  td {{ padding: 10px 12px; border-bottom: 1px solid var(--border); vertical-align: top; }}
  tr:last-child td {{ border-bottom: none; }}
  .badge {{ display: inline-block; padding: 2px 9px; border-radius: 12px; font-size: 0.75rem; font-weight: 600; }}
  .badge-pos {{ background: #c6f6d5; color: #276749; }}
  .badge-neg {{ background: #fed7d7; color: #9b2335; }}
  .badge-ok {{ background: #bee3f8; color: #2a69ac; }}
  .badge-wrong {{ background: #feebc8; color: #c05621; }}
  .stars {{ color: #f6ad55; }}
</style>
</head>
<body>
<header>
  <h1>Amazon Gift Cards — Sentiment Classification</h1>
  <p>Two-class LLM classifier · 100-review batch · Model: cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit</p>
</header>

<div class="container">

  <div class="stats">
    <div class="stat-card">
      <div class="label">Overall Accuracy</div>
      <div class="value" style="color:var(--accent)">{accuracy:.1f}%</div>
      <div class="sub">{correct_count} of {total} correct</div>
    </div>
    <div class="stat-card">
      <div class="label">POSITIVE Accuracy</div>
      <div class="value" style="color:var(--pos)">{pos_correct/len(pos_rows)*100:.1f}%</div>
      <div class="sub">{pos_correct}/{len(pos_rows)} reviews</div>
    </div>
    <div class="stat-card">
      <div class="label">NEGATIVE Accuracy</div>
      <div class="value" style="color:var(--neg)">{neg_correct/len(neg_rows)*100:.1f}%</div>
      <div class="sub">{neg_correct}/{len(neg_rows)} reviews</div>
    </div>
    <div class="stat-card">
      <div class="label">Class Imbalance</div>
      <div class="value" style="color:var(--neutral)">{len(pos_rows)}:{len(neg_rows)}</div>
      <div class="sub">positive : negative</div>
    </div>
  </div>

  <div class="card">
    <h2>Per-Class Performance</h2>
    <div class="bar-row">
      <div class="bar-label">POSITIVE</div>
      <div class="bar-track"><div class="bar-fill" style="width:{pos_correct/len(pos_rows)*100:.1f}%;background:var(--pos)">{pos_correct/len(pos_rows)*100:.1f}%</div></div>
      <div class="bar-pct">{pos_correct}/{len(pos_rows)}</div>
    </div>
    <div class="bar-row">
      <div class="bar-label">NEGATIVE</div>
      <div class="bar-track"><div class="bar-fill" style="width:{neg_correct/len(neg_rows)*100:.1f}%;background:var(--neg)">{neg_correct/len(neg_rows)*100:.1f}%</div></div>
      <div class="bar-pct">{neg_correct}/{len(neg_rows)}</div>
    </div>
    <p style="font-size:0.82rem;color:var(--muted);margin-top:14px;">
      ⚠️ Data is heavily skewed: {len(pos_rows)} of {total} reviews ({len(pos_rows)/total*100:.0f}%) are positive.
      A model that predicts POSITIVE for every review would achieve {len(pos_rows)/total*100:.0f}% accuracy — interpret results with caution.
    </p>
  </div>

  <div class="card">
    <h2>Review Table</h2>
    <div class="filters">
      <button class="filter-btn active" onclick="filterTable('all')">All</button>
      <button class="filter-btn" onclick="filterTable('correct')">✓ Correct</button>
      <button class="filter-btn" onclick="filterTable('wrong')">✗ Wrong</button>
      <button class="filter-btn" onclick="filterTable('POSITIVE')">POSITIVE</button>
      <button class="filter-btn" onclick="filterTable('NEGATIVE')">NEGATIVE</button>
      <span class="count-label" id="count-label">Showing {total} reviews</span>
    </div>
    <table id="review-table">
      <thead>
        <tr>
          <th>#</th><th>Rating</th><th>Ground Truth</th><th>Predicted</th><th>Match</th><th>Title</th>
        </tr>
      </thead>
      <tbody id="table-body"></tbody>
    </table>
  </div>

</div>

<script>
const data = {results_json};

function renderTable(rows) {{
  const tbody = document.getElementById('table-body');
  tbody.innerHTML = '';
  rows.forEach(r => {{
    const tr = document.createElement('tr');
    const stars = '★'.repeat(r.rating) + '☆'.repeat(5 - r.rating);
    tr.innerHTML = `
      <td style="color:var(--muted)">${{r.index + 1}}</td>
      <td><span class="stars">${{stars}}</span></td>
      <td><span class="badge ${{r.ground_truth === 'POSITIVE' ? 'badge-pos' : 'badge-neg'}}">${{r.ground_truth}}</span></td>
      <td><span class="badge ${{r.predicted === 'POSITIVE' ? 'badge-pos' : 'badge-neg'}}">${{r.predicted}}</span></td>
      <td><span class="badge ${{r.correct ? 'badge-ok' : 'badge-wrong'}}">${{r.correct ? '✓' : '✗'}}</span></td>
      <td style="max-width:300px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap" title="${{r.text}}">${{r.title || '(no title)'}}</td>
    `;
    tbody.appendChild(tr);
  }});
  document.getElementById('count-label').textContent = `Showing ${{rows.length}} reviews`;
}}

function filterTable(filter) {{
  document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
  event.target.classList.add('active');
  let filtered = data;
  if (filter === 'correct') filtered = data.filter(r => r.correct);
  else if (filter === 'wrong') filtered = data.filter(r => !r.correct);
  else if (filter === 'POSITIVE') filtered = data.filter(r => r.ground_truth === 'POSITIVE');
  else if (filter === 'NEGATIVE') filtered = data.filter(r => r.ground_truth === 'NEGATIVE');
  renderTable(filtered);
}}

renderTable(data);
</script>
</body>
</html>"""

with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)

print("dashboard.html saved!")

dashboard.html saved!


In [12]:
def classify_sentiment_and_emotion(title: str, text: str) -> dict:
    """Returns both sentiment (POSITIVE/NEGATIVE) and primary emotion."""
    prompt = f"""Classify the following Amazon review.

Return ONLY this exact format on two lines:
SENTIMENT: <POSITIVE or NEGATIVE>
EMOTION: <one of: joy, trust, anticipation, surprise, fear, sadness, disgust, anger>

Rules for sentiment:
- Base your decision only on the title and text. Do not use any star rating.
- If title and text conflict, weight the text more heavily.
- Short, sarcastic, or angry reviews → NEGATIVE. Genuine satisfaction → POSITIVE.

Rules for emotion:
- Pick the single strongest emotion expressed by the reviewer.
- joy = happiness, delight. trust = confidence, reliability.
- anticipation = excitement, looking forward. surprise = unexpected.
- fear = worry, concern. sadness = disappointment, regret.
- disgust = strong dislike. anger = frustration, outrage.

Review Title: {title}
Review Text: {text}"""

    response = client.chat.completions.create(
        model="cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=0,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )

    content = response.choices[0].message.content or ""

    # Parse output
    sentiment = "POSITIVE"
    emotion = "joy"
    for line in content.strip().splitlines():
        if line.startswith("SENTIMENT:"):
            val = line.split(":", 1)[1].strip().upper()
            if "NEGATIVE" in val:
                sentiment = "NEGATIVE"
        elif line.startswith("EMOTION:"):
            val = line.split(":", 1)[1].strip().lower()
            valid = ["joy","trust","anticipation","surprise","fear","sadness","disgust","anger"]
            for e in valid:
                if e in val:
                    emotion = e
                    break
    return {"sentiment": sentiment, "emotion_llm": emotion}

# Quick test
test = classify_sentiment_and_emotion("Waste of money", "This card never worked. Total scam.")
print(test)

{'sentiment': 'NEGATIVE', 'emotion_llm': 'anger'}


In [13]:
!pip install nrclex -q
print("nrclex installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.3 MB/s eta 0:00:00
nrclex installed!


In [14]:
from nrclex import NRCLex
import json

with open("results_100.json") as f:
    results = json.load(f)

EMOTIONS = ["joy","trust","anticipation","surprise","fear","sadness","disgust","anger"]

def get_nrc_emotion(title: str, text: str) -> str:
    """Scores words against NRC lexicon and returns the top emotion."""
    combined = (title + " " + text).lower()
    nrc = NRCLex(combined)
    scores = nrc.affect_frequencies
    # Only keep the 8 primary emotions
    filtered = {e: scores.get(e, 0) for e in EMOTIONS}
    if max(filtered.values()) == 0:
        return "joy"  # default if no emotion words found
    return max(filtered, key=filtered.get)

print("Running LLM + NRC emotion detection on 100 reviews...")
for i, r in enumerate(results):
    # LLM prediction
    llm_result = classify_sentiment_and_emotion(r["title"], r["text"])
    r["emotion_llm"] = llm_result["emotion_llm"]

    # NRC word list prediction
    r["emotion_nrc"] = get_nrc_emotion(r["title"], r["text"])

    # Agreement
    r["emotion_agree"] = (r["emotion_llm"] == r["emotion_nrc"])

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/100...")

# Save updated results
with open("results_100.json", "w") as f:
    json.dump(results, f, indent=2)

# Summary
agree_count = sum(1 for r in results if r["emotion_agree"])
print(f"\n=== Emotion Agreement ===")
print(f"LLM vs NRC agree: {agree_count}/100 ({agree_count}%)")

print(f"\nLLM emotion distribution:")
for e in EMOTIONS:
    count = sum(1 for r in results if r["emotion_llm"] == e)
    if count > 0:
        print(f"  {e}: {count}")

print(f"\nNRC emotion distribution:")
for e in EMOTIONS:
    count = sum(1 for r in results if r["emotion_nrc"] == e)
    if count > 0:
        print(f"  {e}: {count}")

print(f"\nDisagreement examples (first 5):")
for r in [r for r in results if not r["emotion_agree"]][:5]:
    print(f"  [{r['index']+1}] LLM={r['emotion_llm']}, NRC={r['emotion_nrc']} | \"{r['title'][:50]}\"")

Running LLM + NRC emotion detection on 100 reviews...


AttributeError: 'NRCLex' object has no attribute 'affect_frequencies'

In [15]:
from nrclex import NRCLex
test_nrc = NRCLex("love this amazing gift so happy")
print(dir(test_nrc))
print(test_nrc.raw_emotion_scores)

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lexicon__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_build_word_affect', '_compute_top_emotions', '_resolve_lexicon', 'load_raw_text', 'load_token_list']


AttributeError: 'NRCLex' object has no attribute 'raw_emotion_scores'

In [16]:
from nrclex import NRCLex
test_nrc = NRCLex("")
test_nrc.load_raw_text("love this amazing gift so happy")
print(dir(test_nrc))

IsADirectoryError: [Errno 21] Is a directory: '/content'

In [17]:
from nrclex import NRCLex
test_nrc = NRCLex("love this amazing gift so happy")
print([a for a in dir(test_nrc) if not a.startswith('_')])

['load_raw_text', 'load_token_list']


In [18]:
import urllib.request

# Download NRC lexicon directly from GitHub
url = "https://raw.githubusercontent.com/kmike/nrc-emotion-lexicon/master/NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt"
urllib.request.urlretrieve(url, "nrc_lexicon.txt")

# Parse it into a dictionary: word -> list of emotions
nrc = {}
EMOTIONS = ["anger","anticipation","disgust","fear","joy","sadness","surprise","trust"]

with open("nrc_lexicon.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith(";"):
            continue
        parts = line.split("\t")
        if len(parts) != 3:
            continue
        word, emotion, flag = parts
        if emotion in EMOTIONS and flag == "1":
            nrc.setdefault(word, []).append(emotion)

print(f"Loaded NRC lexicon: {len(nrc)} words with emotion associations")
print(f"Sample: {list(nrc.items())[:5]}")

HTTPError: HTTP Error 404: Not Found

In [19]:
import nrclex, os

# Find where nrclex is installed and look for its data file
pkg_dir = os.path.dirname(nrclex.__file__)
print("nrclex package dir:", pkg_dir)
for root, dirs, files in os.walk(pkg_dir):
    for f in files:
        print(os.path.join(root, f))

nrclex package dir: /usr/local/lib/python3.13/dist-packages/nrclex
/usr/local/lib/python3.13/dist-packages/nrclex/__init__.py
/usr/local/lib/python3.13/dist-packages/nrclex/core.py
/usr/local/lib/python3.13/dist-packages/nrclex/__pycache__/core.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/nrclex/__pycache__/__init__.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/nrclex/data/nrc_en.json
/usr/local/lib/python3.13/dist-packages/nrclex/data/__init__.py
/usr/local/lib/python3.13/dist-packages/nrclex/data/__pycache__/__init__.cpython-313.pyc


In [20]:
import json
import re

# Load NRC lexicon directly from the package
with open("/usr/local/lib/python3.13/dist-packages/nrclex/data/nrc_en.json") as f:
    nrc_data = json.load(f)

# Quick peek at structure
first_key = list(nrc_data.keys())[0]
print(f"Sample entry: '{first_key}' -> {nrc_data[first_key]}")
print(f"Total words: {len(nrc_data)}")

EMOTIONS = ["anger","anticipation","disgust","fear","joy","sadness","surprise","trust"]

def get_nrc_emotion(title: str, text: str) -> str:
    """Score words against NRC lexicon and return the top emotion."""
    combined = (title + " " + text).lower()
    words = re.findall(r"[a-z]+", combined)

    scores = {e: 0 for e in EMOTIONS}
    for word in words:
        if word in nrc_data:
            for emotion in nrc_data[word]:
                if emotion in scores:
                    scores[emotion] += 1

    if max(scores.values()) == 0:
        return "joy"  # default if no emotion words found
    return max(scores, key=scores.get)

# Quick test
print("\nTest:")
print(get_nrc_emotion("Love it!!", "This is an amazing gift, so happy!"))
print(get_nrc_emotion("Waste of money", "This card never worked.

SyntaxError: unterminated string literal (detected at line 34) (3365013216.py, line 34)

In [21]:
import json
import re

with open("/usr/local/lib/python3.13/dist-packages/nrclex/data/nrc_en.json") as f:
    nrc_data = json.load(f)

first_key = list(nrc_data.keys())[0]
print(f"Sample: '{first_key}' -> {nrc_data[first_key]}")
print(f"Total words: {len(nrc_data)}")

EMOTIONS = ["anger","anticipation","disgust","fear","joy","sadness","surprise","trust"]

def get_nrc_emotion(title, text):
    combined = (title + " " + text).lower()
    words = re.findall(r"[a-z]+", combined)
    scores = {e: 0 for e in EMOTIONS}
    for word in words:
        if word in nrc_data:
            for emotion in nrc_data[word]:
                if emotion in scores:
                    scores[emotion] += 1
    if max(scores.values()) == 0:
        return "joy"
    return max(scores, key=scores.get)

print(get_nrc_emotion("Love it", "This is an amazing gift so happy"))
print(get_nrc_emotion("Waste of money", "This card never worked total scam"))

Sample: 'abacus' -> ['trust']
Total words: 6468
joy
anger


In [22]:
import json

with open("results_100.json") as f:
    results = json.load(f)

print("Running LLM + NRC emotion detection on 100 reviews...")
for i, r in enumerate(results):
    # LLM emotion
    llm_result = classify_sentiment_and_emotion(r["title"], r["text"])
    r["emotion_llm"] = llm_result["emotion_llm"]

    # NRC word list emotion
    r["emotion_nrc"] = get_nrc_emotion(r["title"], r["text"])

    # Do they agree?
    r["emotion_agree"] = (r["emotion_llm"] == r["emotion_nrc"])

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/100...")

with open("results_100.json", "w") as f:
    json.dump(results, f, indent=2)

agree_count = sum(1 for r in results if r["emotion_agree"])
print(f"\n=== Emotion Agreement ===")
print(f"LLM vs NRC agree: {agree_count}/100 ({agree_count}%)")

print(f"\nLLM emotion distribution:")
for e in EMOTIONS:
    count = sum(1 for r in results if r["emotion_llm"] == e)
    if count > 0:
        print(f"  {e}: {count}")

print(f"\nNRC emotion distribution:")
for e in EMOTIONS:
    count = sum(1 for r in results if r["emotion_nrc"] == e)
    if count > 0:
        print(f"  {e}: {count}")

print(f"\nDisagreement examples (first 5):")
for r in [r for r in results if not r["emotion_agree"]][:5]:
    print(f"  [{r['index']+1}] LLM={r['emotion_llm']}, NRC={r['emotion_nrc']} | title: \"{r['title'][:50]}\"")

Running LLM + NRC emotion detection on 100 reviews...
  Processed 10/100...
  Processed 20/100...
  Processed 30/100...
  Processed 40/100...
  Processed 50/100...
  Processed 60/100...
  Processed 70/100...
  Processed 80/100...
  Processed 90/100...
  Processed 100/100...

=== Emotion Agreement ===
LLM vs NRC agree: 33/100 (33%)

LLM emotion distribution:
  anger: 7
  anticipation: 1
  disgust: 1
  joy: 79
  trust: 12

NRC emotion distribution:
  anger: 2
  anticipation: 60
  disgust: 1
  joy: 36
  sadness: 1

Disagreement examples (first 5):
  [1] LLM=joy, NRC=anticipation | title: "Great gift"
  [2] LLM=joy, NRC=anticipation | title: "amazon gift card"
  [5] LLM=anger, NRC=joy | title: "Not $10 Gift Cards"
  [7] LLM=joy, NRC=anticipation | title: "Easy gift"
  [8] LLM=joy, NRC=anticipation | title: "Great gift"


In [23]:
import urllib.request
import gzip
import json
import random

random.seed(42)  # fixed seed for reproducibility

url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Gift_Cards.jsonl.gz"

def rating_to_3class(rating):
    if rating >= 4:
        return "POSITIVE"
    elif rating == 3:
        return "NEUTRAL"
    else:
        return "NEGATIVE"

print("Scanning full dataset to collect balanced samples...")
buckets = {"POSITIVE": [], "NEUTRAL": [], "NEGATIVE": []}

with urllib.request.urlopen(url) as response:
    with gzip.GzipFile(fileobj=response) as f:
        for line in f:
            r = json.loads(line)
            label = rating_to_3class(r["rating"])
            if len(buckets[label]) < 200:
                buckets[label].append(r)
            if all(len(v) >= 200 for v in buckets.values()):
                break

for label, items in buckets.items():
    print(f"  {label}: collected {len(items)} candidates")

# Sample 50 from each
balanced = []
for label, items in buckets.items():
    sample = random.sample(items, 50)
    for r in sample:
        r["true_label"] = label
    balanced.extend(sample)

random.shuffle(balanced)
print(f"\nBalanced set: {len(balanced)} reviews (50 per class)")

Scanning full dataset to collect balanced samples...
  POSITIVE: collected 200 candidates
  NEUTRAL: collected 200 candidates
  NEGATIVE: collected 200 candidates

Balanced set: 150 reviews (50 per class)


In [24]:
def classify_3class(title: str, text: str) -> dict:
    """Classifies review as POSITIVE, NEUTRAL, or NEGATIVE + primary emotion."""
    prompt = f"""Classify the following Amazon review.

Return ONLY this exact format on two lines:
SENTIMENT: <POSITIVE or NEUTRAL or NEGATIVE>
EMOTION: <one of: joy, trust, anticipation, surprise, fear, sadness, disgust, anger>

Rules for sentiment:
- Base your decision ONLY on the title and text. Never use a star rating.
- POSITIVE: genuine satisfaction, praise, happiness with the product.
- NEUTRAL: mixed feelings, neither clearly happy nor unhappy, or purely factual.
- NEGATIVE: disappointment, frustration, complaint, or strong dissatisfaction.

Rules for emotion:
- Pick the single strongest emotion expressed by the reviewer.

Review Title: {title}
Review Text: {text}"""

    response = client.chat.completions.create(
        model="cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=0,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )

    content = response.choices[0].message.content or ""
    sentiment = "POSITIVE"
    emotion = "joy"
    for line in content.strip().splitlines():
        if line.startswith("SENTIMENT:"):
            val = line.split(":", 1)[1].strip().upper()
            if "NEGATIVE" in val:
                sentiment = "NEGATIVE"
            elif "NEUTRAL" in val:
                sentiment = "NEUTRAL"
        elif line.startswith("EMOTION:"):
            val = line.split(":", 1)[1].strip().lower()
            for e in EMOTIONS:
                if e in val:
                    emotion = e
                    break
    return {"sentiment": sentiment, "emotion_llm": emotion}

print("Classifying 150 balanced reviews (3-class)...")
balanced_results = []
for i, r in enumerate(balanced):
    title = r.get("title") or ""
    text = r.get("text") or ""
    pred = classify_3class(title, text)
    balanced_results.append({
        "index": i,
        "rating": r["rating"],
        "true_label": r["true_label"],
        "predicted": pred["sentiment"],
        "correct": pred["sentiment"] == r["true_label"],
        "emotion_llm": pred["emotion_llm"],
        "emotion_nrc": get_nrc_emotion(title, text),
        "title": title,
        "text": text[:200]
    })
    if (i + 1) % 15 == 0:
        print(f"  Processed {i+1}/150...")

with open("results_balanced.json", "w") as f:
    json.dump(balanced_results, f, indent=2)

# Accuracy summary
total = len(balanced_results)
correct = sum(1 for r in balanced_results if r["correct"])
print(f"\n=== 3-Class Balanced Results ===")
print(f"Overall accuracy: {correct}/{total} = {correct/total*100:.1f}%")

for label in ["POSITIVE", "NEUTRAL", "NEGATIVE"]:
    rows = [r for r in balanced_results if r["true_label"] == label]
    c = sum(1 for r in rows if r["correct"])
    print(f"  {label}: {c}/{len(rows)} ({c/len(rows)*100:.1f}%)")

# Confusion matrix
print(f"\nConfusion matrix (rows=true, cols=predicted):")
print(f"{'':12} {'POSITIVE':>10} {'NEUTRAL':>10} {'NEGATIVE':>10}")
for true in ["POSITIVE","NEUTRAL","NEGATIVE"]:
    row = [r for r in balanced_results if r["true_label"] == true]
    counts = {p: sum(1 for r in row if r["predicted"] == p) for p in ["POSITIVE","NEUTRAL","NEGATIVE"]}
    print(f"{true:12} {counts['POSITIVE']:>10} {counts['NEUTRAL']:>10} {counts['NEGATIVE']:>10}")

Classifying 150 balanced reviews (3-class)...
  Processed 15/150...
  Processed 30/150...
  Processed 45/150...
  Processed 60/150...
  Processed 75/150...
  Processed 90/150...
  Processed 105/150...
  Processed 120/150...
  Processed 135/150...
  Processed 150/150...

=== 3-Class Balanced Results ===
Overall accuracy: 111/150 = 74.0%
  POSITIVE: 49/50 (98.0%)
  NEUTRAL: 16/50 (32.0%)
  NEGATIVE: 46/50 (92.0%)

Confusion matrix (rows=true, cols=predicted):
               POSITIVE    NEUTRAL   NEGATIVE
POSITIVE             49          1          0
NEUTRAL               3         16         31
NEGATIVE              0          4         46


In [26]:
import json

with open("results_100.json") as f:
    r2 = json.load(f)
with open("results_balanced.json") as f:
    r3 = json.load(f)

# Pre-compute all stats
# 2-class stats
t2 = len(r2)
c2 = sum(1 for r in r2 if r["correct"])
pos2 = [r for r in r2 if r["ground_truth"]=="POSITIVE"]
neg2 = [r for r in r2 if r["ground_truth"]=="NEGATIVE"]
pc2 = sum(1 for r in pos2 if r["correct"])
nc2 = sum(1 for r in neg2 if r["correct"])

# 3-class stats
t3 = len(r3)
c3 = sum(1 for r in r3 if r["correct"])
labels3 = ["POSITIVE","NEUTRAL","NEGATIVE"]
acc3 = {}
for lb in labels3:
    rows = [r for r in r3 if r["true_label"]==lb]
    acc3[lb] = sum(1 for r in rows if r["correct"])

# Confusion matrix
cm = {}
for true in labels3:
    cm[true] = {}
    for pred in labels3:
        cm[true][pred] = sum(1 for r in r3 if r["true_label"]==true and r["predicted"]==pred)

# Star distribution from 2-class batch
star_dist = {}
for r in r2:
    s = int(r["rating"])
    star_dist[s] = star_dist.get(s, 0) + 1

# Emotion distributions
llm_emotions = {}
nrc_emotions = {}
for r in r3:
    llm_emotions[r["emotion_llm"]] = llm_emotions.get(r["emotion_llm"], 0) + 1
    nrc_emotions[r["emotion_nrc"]] = nrc_emotions.get(r["emotion_nrc"], 0) + 1

r2_json = json.dumps(r2)
r3_json = json.dumps(r3)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Amazon Review Sentiment Dashboard</title>
<style>
  :root {{
    --bg:#f8f9fc;--card:#fff;--border:#e2e8f0;--text:#1a202c;--muted:#718096;
    --pos:#38a169;--neg:#e53e3e;--neu:#d69e2e;--accent:#3b82f6;--accent-dark:#2563eb;
  }}
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{font-family:'Segoe UI',system-ui,sans-serif;background:var(--bg);color:var(--text)}}
  header{{background:var(--accent-dark);color:white;padding:28px 40px}}
  header h1{{font-size:1.6rem;font-weight:700}}
  header p{{font-size:0.9rem;opacity:0.8;margin-top:4px}}
  nav{{display:flex;gap:0;background:#1e40af;padding:0 40px}}
  nav button{{padding:10px 20px;background:none;border:none;color:rgba(255,255,255,0.7);cursor:pointer;font-size:0.88rem;border-bottom:3px solid transparent}}
  nav button.active{{color:white;border-bottom-color:white}}
  .tab{{display:none}}.tab.active{{display:block}}
  .container{{max-width:1100px;margin:32px auto;padding:0 24px}}
  .stats{{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:28px}}
  .stat-card{{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:20px 24px}}
  .stat-card .label{{font-size:0.78rem;color:var(--muted);text-transform:uppercase;letter-spacing:0.05em}}
  .stat-card .value{{font-size:2rem;font-weight:700;margin-top:6px}}
  .stat-card .sub{{font-size:0.82rem;color:var(--muted);margin-top:2px}}
  .card{{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:24px;margin-bottom:24px}}
  .card h2{{font-size:1rem;font-weight:600;margin-bottom:16px}}
  .grid2{{display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:24px}}
  .bar-row{{display:flex;align-items:center;gap:12px;margin-bottom:10px}}
  .bar-label{{width:100px;font-size:0.83rem;font-weight:600;flex-shrink:0}}
  .bar-track{{flex:1;background:var(--bg);border-radius:6px;height:22px;overflow:hidden}}
  .bar-fill{{height:100%;border-radius:6px;display:flex;align-items:center;padding-left:8px;font-size:0.78rem;color:white;font-weight:600}}
  .bar-pct{{font-size:0.83rem;color:var(--muted);width:60px;text-align:right;flex-shrink:0}}
  .cm-table{{width:100%;border-collapse:collapse;font-size:0.85rem}}
  .cm-table th,.cm-table td{{padding:10px 14px;text-align:center;border:1px solid var(--border)}}
  .cm-table th{{background:var(--bg);font-weight:600;color:var(--muted);font-size:0.78rem}}
  .cm-diag{{background:#ebf8ff;font-weight:700;color:#2b6cb0}}
  .cm-err{{background:#fff5f5;color:#c53030}}
  .filters{{display:flex;gap:10px;margin-bottom:14px;flex-wrap:wrap;align-items:center}}
  .filter-btn{{padding:6px 14px;border:1px solid var(--border);border-radius:20px;background:white;cursor:pointer;font-size:0.83rem}}
  .filter-btn.active{{background:var(--accent);color:white;border-color:var(--accent)}}
  .count-label{{margin-left:auto;font-size:0.83rem;color:var(--muted)}}
  table{{width:100%;border-collapse:collapse;font-size:0.85rem}}
  th{{text-align:left;padding:10px 12px;border-bottom:2px solid var(--border);color:var(--muted);font-weight:600;font-size:0.78rem;text-transform:uppercase}}
  td{{padding:10px 12px;border-bottom:1px solid var(--border);vertical-align:top}}
  tr:last-child td{{border-bottom:none}}
  .badge{{display:inline-block;padding:2px 9px;border-radius:12px;font-size:0.75rem;font-weight:600}}
  .badge-pos{{background:#c6f6d5;color:#276749}}
  .badge-neg{{background:#fed7d7;color:#9b2335}}
  .badge-neu{{background:#fefcbf;color:#744210}}
  .badge-ok{{background:#bee3f8;color:#2a69ac}}
  .badge-wrong{{background:#feebc8;color:#c05621}}
  .stars{{color:#f6ad55}}
  .warn{{font-size:0.82rem;color:var(--muted);margin-top:14px;padding:10px 14px;background:#fffbeb;border-left:3px solid #f6ad55;border-radius:4px}}
</style>
</head>
<body>
<header>
  <h1>Amazon Gift Cards — Sentiment &amp; Emotion Dashboard</h1>
  <p>LLM classifier · Model: cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit · Amazon Reviews 2023</p>
</header>
<nav>
  <button class="active" onclick="showTab('tab-overview',this)">Overview</button>
  <button onclick="showTab('tab-3class',this)">3-Class Analysis</button>
  <button onclick="showTab('tab-emotions',this)">Emotions</button>
  <button onclick="showTab('tab-reviews',this)">Review Table</button>
</nav>

<!-- TAB 1: OVERVIEW -->
<div id="tab-overview" class="tab active">
<div class="container">
  <div class="stats">
    <div class="stat-card">
      <div class="label">2-Class Accuracy</div>
      <div class="value" style="color:var(--accent)">{c2/t2*100:.1f}%</div>
      <div class="sub">{c2}/{t2} · imbalanced batch</div>
    </div>
    <div class="stat-card">
      <div class="label">3-Class Accuracy</div>
      <div class="value" style="color:var(--neu)">{c3/t3*100:.1f}%</div>
      <div class="sub">{c3}/{t3} · balanced batch</div>
    </div>
    <div class="stat-card">
      <div class="label">POSITIVE Acc (3-class)</div>
      <div class="value" style="color:var(--pos)">{acc3['POSITIVE']/50*100:.1f}%</div>
      <div class="sub">{acc3['POSITIVE']}/50 reviews</div>
    </div>
    <div class="stat-card">
      <div class="label">NEUTRAL Acc (3-class)</div>
      <div class="value" style="color:var(--neg)">{acc3['NEUTRAL']/50*100:.1f}%</div>
      <div class="sub">{acc3['NEUTRAL']}/50 · hardest class</div>
    </div>
  </div>

  <div class="grid2">
    <div class="card">
      <h2>Star Rating Distribution (first 100 reviews)</h2>
      {''.join(f"""<div class="bar-row">
        <div class="bar-label">{'★'*s}{'☆'*(5-s)}</div>
        <div class="bar-track"><div class="bar-fill" style="width:{star_dist.get(s,0)}%;background:var(--{'pos' if s>=4 else 'neg' if s<=2 else 'neu'})">{star_dist.get(s,0)}</div></div>
        <div class="bar-pct">{star_dist.get(s,0)}</div>
      </div>""" for s in [5,4,3,2,1])}
      <div class="warn">⚠️ {len(pos2)} of {t2} reviews ({len(pos2)/t2*100:.0f}%) are positive — a naive "always POSITIVE" baseline would score {len(pos2)/t2*100:.0f}%.</div>
    </div>
    <div class="card">
      <h2>Per-Class Accuracy (3-class balanced)</h2>
      {''.join(f"""<div class="bar-row">
        <div class="bar-label">{lb}</div>
        <div class="bar-track"><div class="bar-fill" style="width:{acc3[lb]/50*100:.1f}%;background:var(--{'pos' if lb=='POSITIVE' else 'neg' if lb=='NEGATIVE' else 'neu'})">{acc3[lb]/50*100:.1f}%</div></div>
        <div class="bar-pct">{acc3[lb]}/50</div>
      </div>""" for lb in labels3)}
      <div class="warn">⚠️ NEUTRAL (3★) reviews are the hardest — the model collapses most of them into NEGATIVE.</div>
    </div>
  </div>
</div>
</div>

<!-- TAB 2: 3-CLASS ANALYSIS -->
<div id="tab-3class" class="tab">
<div class="container">
  <div class="card">
    <h2>Confusion Matrix — 3-Class Balanced Run (150 reviews, 50 per class)</h2>
    <table class="cm-table">
      <thead>
        <tr><th>True \\ Predicted</th><th>POSITIVE</th><th>NEUTRAL</th><th>NEGATIVE</th></tr>
      </thead>
      <tbody>
        {''.join(f"""<tr><th>{true}</th>{''.join(f'<td class="{"cm-diag" if true==pred else "cm-err" if cm[true][pred]>0 else ""}">{cm[true][pred]}</td>' for pred in labels3)}</tr>""" for true in labels3)}
      </tbody>
    </table>
    <div class="warn" style="margin-top:16px">
      Key finding: {cm['NEUTRAL']['NEGATIVE']} of 50 NEUTRAL (3★) reviews were predicted NEGATIVE — the model collapses ambiguous reviews into the negative class.
      Only {acc3['NEUTRAL']} of 50 NEUTRAL reviews were correctly identified.
    </div>
  </div>
</div>
</div>

<!-- TAB 3: EMOTIONS -->
<div id="tab-emotions" class="tab">
<div class="container">
  <div class="grid2">
    <div class="card">
      <h2>LLM Emotion Distribution (150 reviews)</h2>
      {''.join(f"""<div class="bar-row">
        <div class="bar-label">{e}</div>
        <div class="bar-track"><div class="bar-fill" style="width:{llm_emotions.get(e,0)/1.5:.0f}%;background:var(--accent)">{llm_emotions.get(e,0)}</div></div>
        <div class="bar-pct">{llm_emotions.get(e,0)}</div>
      </div>""" for e in EMOTIONS if llm_emotions.get(e,0)>0)}
    </div>
    <div class="card">
      <h2>NRC Word-List Emotion Distribution (150 reviews)</h2>
      {''.join(f"""<div class="bar-row">
        <div class="bar-label">{e}</div>
        <div class="bar-track"><div class="bar-fill" style="width:{nrc_emotions.get(e,0)/1.5:.0f}%;background:#805ad5">{nrc_emotions.get(e,0)}</div></div>
        <div class="bar-pct">{nrc_emotions.get(e,0)}</div>
      </div>""" for e in EMOTIONS if nrc_emotions.get(e,0)>0)}
    </div>
  </div>
  <div class="card">
    <h2>Why do they diverge?</h2>
    <p style="font-size:0.9rem;line-height:1.7;color:var(--text)">
      The LLM reads the <em>intent</em> of the review — a reviewer saying "perfect gift!" registers as <strong>joy</strong>.
      The NRC lexicon scores individual words: "gift", "give", "perfect", "love" are all tagged <strong>anticipation</strong> in the NRC database,
      pulling the distribution heavily toward anticipation even for clearly happy reviews.
      The two methods agree on {sum(1 for r in r3 if r['emotion_llm']==r['emotion_nrc'])}/150 reviews ({sum(1 for r in r3 if r['emotion_llm']==r['emotion_nrc'])/1.5:.0f}%).
    </p>
  </div>
</div>
</div>

<!-- TAB 4: REVIEW TABLE -->
<div id="tab-reviews" class="tab">
<div class="container">
  <div class="card">
    <h2>Review Table — 3-Class Balanced Run</h2>
    <div class="filters">
      <button class="filter-btn active" onclick="filterTable('all',this)">All</button>
      <button class="filter-btn" onclick="filterTable('correct',this)">✓ Correct</button>
      <button class="filter-btn" onclick="filterTable('wrong',this)">✗ Wrong</button>
      <button class="filter-btn" onclick="filterTable('POSITIVE',this)">POSITIVE</button>
      <button class="filter-btn" onclick="filterTable('NEUTRAL',this)">NEUTRAL</button>
      <button class="filter-btn" onclick="filterTable('NEGATIVE',this)">NEGATIVE</button>
      <span class="count-label" id="count-label">Showing 150 reviews</span>
    </div>
    <table><thead><tr>
      <th>#</th><th>Rating</th><th>True Label</th><th>Predicted</th><th>Match</th><th>LLM Emotion</th><th>NRC Emotion</th><th>Title</th>
    </tr></thead><tbody id="table-body"></tbody></table>
  </div>
</div>
</div>

<script>
const data3 = {r3_json};
const BADGE = {{
  POSITIVE:'badge-pos', NEUTRAL:'badge-neu', NEGATIVE:'badge-neg'
}};
function showTab(id, btn) {{
  document.querySelectorAll('.tab').forEach(t=>t.classList.remove('active'));
  document.querySelectorAll('nav button').forEach(b=>b.classList.remove('active'));
  document.getElementById(id).classList.add('active');
  btn.classList.add('active');
}}
function renderTable(rows) {{
  const tbody = document.getElementById('table-body');
  tbody.innerHTML = '';
  rows.forEach(r => {{
    const stars = '★'.repeat(r.rating)+'☆'.repeat(5-r.rating);
    const tr = document.createElement('tr');
    tr.innerHTML = `
      <td style="color:var(--muted)">${{r.index+1}}</td>
      <td><span class="stars">${{stars}}</span></td>
      <td><span class="badge ${{BADGE[r.true_label]}}">${{r.true_label}}</span></td>
      <td><span class="badge ${{BADGE[r.predicted]}}">${{r.predicted}}</span></td>
      <td><span class="badge ${{r.correct?'badge-ok':'badge-wrong'}}">${{r.correct?'✓':'✗'}}</span></td>
      <td style="font-size:0.8rem">${{r.emotion_llm}}</td>
      <td style="font-size:0.8rem">${{r.emotion_nrc}}</td>
      <td style="max-width:250px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;font-size:0.82rem" title="${{r.text}}">${{r.title||'(no title)'}}</td>`;
    tbody.appendChild(tr);
  }});
  document.getElementById('count-label').textContent=`Showing ${{rows.length}} reviews`;
}}
function filterTable(filter, btn) {{
  document.querySelectorAll('.filter-btn').forEach(b=>b.classList.remove('active'));
  btn.classList.add('active');
  let filtered = data3;
  if(filter==='correct') filtered=data3.filter(r=>r.correct);
  else if(filter==='wrong') filtered=data3.filter(r=>!r.correct);
  else if(['POSITIVE','NEUTRAL','NEGATIVE'].includes(filter)) filtered=data3.filter(r=>r.true_label===filter);
  renderTable(filtered);
}}
renderTable(data3);
</script>
</body>
</html>"""

with open("dashboard_final.html","w",encoding="utf-8") as f:
    f.write(html)
print("dashboard_final.html saved!")

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 150)

In [27]:
import json

with open("results_100.json") as f:
    r2 = json.load(f)
with open("results_balanced.json") as f:
    r3 = json.load(f)

# Stats
t2, c2 = len(r2), sum(1 for r in r2 if r["correct"])
pos2 = [r for r in r2 if r["ground_truth"]=="POSITIVE"]
neg2 = [r for r in r2 if r["ground_truth"]=="NEGATIVE"]
t3, c3 = len(r3), sum(1 for r in r3 if r["correct"])
labels3 = ["POSITIVE","NEUTRAL","NEGATIVE"]
acc3 = {lb: sum(1 for r in r3 if r["true_label"]==lb and r["correct"]) for lb in labels3}
cm = {true: {pred: sum(1 for r in r3 if r["true_label"]==true and r["predicted"]==pred) for pred in labels3} for true in labels3}
star_dist = {}
for r in r2:
    s = int(r["rating"])
    star_dist[s] = star_dist.get(s, 0) + 1
EMOTIONS = ["anger","anticipation","disgust","fear","joy","sadness","surprise","trust"]
llm_emo = {e: sum(1 for r in r3 if r.get("emotion_llm")==e) for e in EMOTIONS}
nrc_emo = {e: sum(1 for r in r3 if r.get("emotion_nrc")==e) for e in EMOTIONS}
emo_agree = sum(1 for r in r3 if r.get("emotion_llm")==r.get("emotion_nrc"))

# Build HTML sections separately
star_bars = ""
for s in [5,4,3,2,1]:
    cnt = star_dist.get(s, 0)
    color = "var(--pos)" if s>=4 else ("var(--neu)" if s==3 else "var(--neg)")
    stars_str = "★"*s + "☆"*(5-s)
    star_bars += f'<div class="bar-row"><div class="bar-label">{stars_str}</div><div class="bar-track"><div class="bar-fill" style="width:{cnt}%;background:{color}">{cnt}</div></div><div class="bar-pct">{cnt}</div></div>\n'

acc_bars = ""
for lb in labels3:
    pct = acc3[lb]/50*100
    color = "var(--pos)" if lb=="POSITIVE" else ("var(--neg)" if lb=="NEGATIVE" else "var(--neu)")
    acc_bars += f'<div class="bar-row"><div class="bar-label">{lb}</div><div class="bar-track"><div class="bar-fill" style="width:{pct:.1f}%;background:{color}">{pct:.1f}%</div></div><div class="bar-pct">{acc3[lb]}/50</div></div>\n'

cm_rows = ""
for true in labels3:
    cm_rows += f"<tr><th>{true}</th>"
    for pred in labels3:
        val = cm[true][pred]
        cls = "cm-diag" if true==pred else ("cm-err" if val>0 else "")
        cm_rows += f'<td class="{cls}">{val}</td>'
    cm_rows += "</tr>\n"

llm_bars = ""
for e in EMOTIONS:
    cnt = llm_emo.get(e, 0)
    if cnt > 0:
        llm_bars += f'<div class="bar-row"><div class="bar-label">{e}</div><div class="bar-track"><div class="bar-fill" style="width:{cnt/1.5:.0f}%;background:var(--accent)">{cnt}</div></div><div class="bar-pct">{cnt}</div></div>\n'

nrc_bars = ""
for e in EMOTIONS:
    cnt = nrc_emo.get(e, 0)
    if cnt > 0:
        nrc_bars += f'<div class="bar-row"><div class="bar-label">{e}</div><div class="bar-track"><div class="bar-fill" style="width:{cnt/1.5:.0f}%;background:#805ad5">{cnt}</div></div><div class="bar-pct">{cnt}</div></div>\n'

r3_json = json.dumps(r3)

print("All stats computed. Run next cell to generate HTML.")
print(f"2-class: {c2}/{t2} = {c2/t2*100:.1f}%")
print(f"3-class: {c3}/{t3} = {c3/t3*100:.1f}%")
print(f"Emotion agreement: {emo_agree}/150")

All stats computed. Run next cell to generate HTML.
2-class: 98/100 = 98.0%
3-class: 111/150 = 74.0%
Emotion agreement: 27/150


In [28]:
html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Amazon Review Sentiment Dashboard</title>
<style>
  :root {--bg:#f8f9fc;--card:#fff;--border:#e2e8f0;--text:#1a202c;--muted:#718096;
    --pos:#38a169;--neg:#e53e3e;--neu:#d69e2e;--accent:#3b82f6;--accent-dark:#2563eb;}
  *{box-sizing:border-box;margin:0;padding:0}
  body{font-family:'Segoe UI',system-ui,sans-serif;background:var(--bg);color:var(--text)}
  header{background:var(--accent-dark);color:white;padding:28px 40px}
  header h1{font-size:1.6rem;font-weight:700}
  header p{font-size:0.9rem;opacity:0.8;margin-top:4px}
  nav{display:flex;background:#1e40af;padding:0 40px}
  nav button{padding:10px 20px;background:none;border:none;color:rgba(255,255,255,0.7);cursor:pointer;font-size:0.88rem;border-bottom:3px solid transparent}
  nav button.active{color:white;border-bottom-color:white}
  .tab{display:none}.tab.active{display:block}
  .container{max-width:1100px;margin:32px auto;padding:0 24px}
  .stats{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:28px}
  .stat-card{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:20px 24px}
  .stat-card .label{font-size:0.78rem;color:var(--muted);text-transform:uppercase;letter-spacing:0.05em}
  .stat-card .value{font-size:2rem;font-weight:700;margin-top:6px}
  .stat-card .sub{font-size:0.82rem;color:var(--muted);margin-top:2px}
  .card{background:var(--card);border:1px solid var(--border);border-radius:10px;padding:24px;margin-bottom:24px}
  .card h2{font-size:1rem;font-weight:600;margin-bottom:16px}
  .grid2{display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:24px}
  .bar-row{display:flex;align-items:center;gap:12px;margin-bottom:10px}
  .bar-label{width:110px;font-size:0.83rem;font-weight:600;flex-shrink:0}
  .bar-track{flex:1;background:var(--bg);border-radius:6px;height:22px;overflow:hidden}
  .bar-fill{height:100%;border-radius:6px;display:flex;align-items:center;padding-left:8px;font-size:0.78rem;color:white;font-weight:600;min-width:2px}
  .bar-pct{font-size:0.83rem;color:var(--muted);width:50px;text-align:right;flex-shrink:0}
  .cm-table{width:100%;border-collapse:collapse;font-size:0.88rem}
  .cm-table th,.cm-table td{padding:12px 16px;text-align:center;border:1px solid var(--border)}
  .cm-table th{background:var(--bg);font-weight:600;color:var(--muted);font-size:0.78rem}
  .cm-diag{background:#ebf8ff;font-weight:700;color:#2b6cb0;font-size:1.1rem}
  .cm-err{background:#fff5f5;color:#c53030}
  .filters{display:flex;gap:10px;margin-bottom:14px;flex-wrap:wrap;align-items:center}
  .filter-btn{padding:6px 14px;border:1px solid var(--border);border-radius:20px;background:white;cursor:pointer;font-size:0.83rem}
  .filter-btn.active{background:var(--accent);color:white;border-color:var(--accent)}
  .count-label{margin-left:auto;font-size:0.83rem;color:var(--muted)}
  table{width:100%;border-collapse:collapse;font-size:0.85rem}
  th{text-align:left;padding:10px 12px;border-bottom:2px solid var(--border);color:var(--muted);font-weight:600;font-size:0.78rem;text-transform:uppercase}
  td{padding:10px 12px;border-bottom:1px solid var(--border);vertical-align:top}
  tr:last-child td{border-bottom:none}
  .badge{display:inline-block;padding:2px 9px;border-radius:12px;font-size:0.75rem;font-weight:600}
  .badge-pos{background:#c6f6d5;color:#276749}
  .badge-neg{background:#fed7d7;color:#9b2335}
  .badge-neu{background:#fefcbf;color:#744210}
  .badge-ok{background:#bee3f8;color:#2a69ac}
  .badge-wrong{background:#feebc8;color:#c05621}
  .stars{color:#f6ad55}
  .warn{font-size:0.82rem;color:var(--muted);margin-top:14px;padding:10px 14px;background:#fffbeb;border-left:3px solid #f6ad55;border-radius:4px;line-height:1.6}
</style>
</head>
<body>
<header>
  <h1>Amazon Gift Cards — Sentiment &amp; Emotion Dashboard</h1>
  <p>LLM classifier · Model: cyankiwi/Qwen3.6-35B-A3B-AWQ-4bit · Amazon Reviews 2023</p>
</header>
<nav>
  <button class="active" onclick="showTab('tab-overview',this)">Overview</button>
  <button onclick="showTab('tab-3class',this)">3-Class Analysis</button>
  <button onclick="showTab('tab-emotions',this)">Emotions</button>
  <button onclick="showTab('tab-reviews',this)">Review Table</button>
</nav>
""" + f"""
<div id="tab-overview" class="tab active"><div class="container">
  <div class="stats">
    <div class="stat-card"><div class="label">2-Class Accuracy</div><div class="value" style="color:var(--accent)">{c2/t2*100:.1f}%</div><div class="sub">{c2}/{t2} · imbalanced batch</div></div>
    <div class="stat-card"><div class="label">3-Class Accuracy</div><div class="value" style="color:var(--neu)">{c3/t3*100:.1f}%</div><div class="sub">{c3}/{t3} · balanced batch</div></div>
    <div class="stat-card"><div class="label">POSITIVE Accuracy</div><div class="value" style="color:var(--pos)">{acc3['POSITIVE']/50*100:.1f}%</div><div class="sub">{acc3['POSITIVE']}/50 reviews</div></div>
    <div class="stat-card"><div class="label">NEUTRAL Accuracy</div><div class="value" style="color:var(--neg)">{acc3['NEUTRAL']/50*100:.1f}%</div><div class="sub">{acc3['NEUTRAL']}/50 · hardest class</div></div>
  </div>
  <div class="grid2">
    <div class="card"><h2>Star Rating Distribution (first 100 reviews)</h2>
""" + star_bars + f"""
    <div class="warn">⚠️ {len(pos2)} of {t2} reviews ({len(pos2)/t2*100:.0f}%) are positive — a naive "always POSITIVE" baseline would already score {len(pos2)/t2*100:.0f}%.</div>
    </div>
    <div class="card"><h2>Per-Class Accuracy (3-class balanced)</h2>
""" + acc_bars + """
    <div class="warn">⚠️ NEUTRAL (3★) reviews are the hardest class — the model collapses most of them into NEGATIVE.</div>
    </div>
  </div>
</div></div>

<div id="tab-3class" class="tab"><div class="container">
  <div class="card"><h2>Confusion Matrix — 3-Class Balanced Run (150 reviews, 50 per class)</h2>
  <table class="cm-table">
    <thead><tr><th>True \\ Predicted</th><th>POSITIVE</th><th>NEUTRAL</th><th>NEGATIVE</th></tr></thead>
    <tbody>
""" + cm_rows + f"""
    </tbody>
  </table>
  <div class="warn">Key finding: {cm['NEUTRAL']['NEGATIVE']} of 50 NEUTRAL (3★) reviews were predicted NEGATIVE.
  The model treats ambiguous reviews as negative. Only {acc3['NEUTRAL']} of 50 NEUTRAL reviews were correctly identified.</div>
  </div>
</div></div>

<div id="tab-emotions" class="tab"><div class="container">
  <div class="grid2">
    <div class="card"><h2>LLM Emotion Distribution (150 reviews)</h2>
""" + llm_bars + """
    </div>
    <div class="card"><h2>NRC Word-List Emotion Distribution (150 reviews)</h2>
""" + nrc_bars + f"""
    </div>
  </div>
  <div class="card"><h2>Why do they diverge?</h2>
  <p style="font-size:0.9rem;line-height:1.7">
    The LLM reads the <em>intent</em> of the review — "perfect gift!" registers as <strong>joy</strong>.
    The NRC lexicon scores individual words: "gift", "give", "perfect" are tagged as <strong>anticipation</strong> in the NRC database,
    skewing the word-list results heavily. The two methods agree on only {emo_agree}/150 reviews ({emo_agree/1.5:.0f}%).
  </p>
  </div>
</div></div>

<div id="tab-reviews" class="tab"><div class="container">
  <div class="card"><h2>Review Table — 3-Class Balanced Run</h2>
  <div class="filters">
    <button class="filter-btn active" onclick="filterTable('all',this)">All</button>
    <button class="filter-btn" onclick="filterTable('correct',this)">✓ Correct</button>
    <button class="filter-btn" onclick="filterTable('wrong',this)">✗ Wrong</button>
    <button class="filter-btn" onclick="filterTable('POSITIVE',this)">POSITIVE</button>
    <button class="filter-btn" onclick="filterTable('NEUTRAL',this)">NEUTRAL</button>
    <button class="filter-btn" onclick="filterTable('NEGATIVE',this)">NEGATIVE</button>
    <span class="count-label" id="count-label">Showing 150 reviews</span>
  </div>
  <table><thead><tr>
    <th>#</th><th>Rating</th><th>True</th><th>Predicted</th><th>Match</th><th>LLM Emo</th><th>NRC Emo</th><th>Title</th>
  </tr></thead><tbody id="table-body"></tbody></table>
  </div>
</div></div>

<script>
const data3 = """ + r3_json + """;
const BADGE = {POSITIVE:'badge-pos',NEUTRAL:'badge-neu',NEGATIVE:'badge-neg'};
function showTab(id,btn){
  document.querySelectorAll('.tab').forEach(t=>t.classList.remove('active'));
  document.querySelectorAll('nav button').forEach(b=>b.classList.remove('active'));
  document.getElementById(id).classList.add('active');
  btn.classList.add('active');
}
function renderTable(rows){
  const tbody=document.getElementById('table-body');
  tbody.innerHTML='';
  rows.forEach(r=>{
    const stars='★'.repeat(r.rating)+'☆'.repeat(5-r.rating);
    const tr=document.createElement('tr');
    tr.innerHTML=`
      <td style="color:var(--muted)">${r.index+1}</td>
      <td><span class="stars">${stars}</span></td>
      <td><span class="badge ${BADGE[r.true_label]}">${r.true_label}</span></td>
      <td><span class="badge ${BADGE[r.predicted]}">${r.predicted}</span></td>
      <td><span class="badge ${r.correct?'badge-ok':'badge-wrong'}">${r.correct?'✓':'✗'}</span></td>
      <td style="font-size:0.8rem">${r.emotion_llm||''}</td>
      <td style="font-size:0.8rem">${r.emotion_nrc||''}</td>
      <td style="max-width:220px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;font-size:0.82rem" title="${r.text}">${r.title||'(no title)'}</td>`;
    tbody.appendChild(tr);
  });
  document.getElementById('count-label').textContent=`Showing ${rows.length} reviews`;
}
function filterTable(filter,btn){
  document.querySelectorAll('.filter-btn').forEach(b=>b.classList.remove('active'));
  btn.classList.add('active');
  let filtered=data3;
  if(filter==='correct') filtered=data3.filter(r=>r.correct);
  else if(filter==='wrong') filtered=data3.filter(r=>!r.correct);
  else if(['POSITIVE','NEUTRAL','NEGATIVE'].includes(filter)) filtered=data3.filter(r=>r.true_label===filter);
  renderTable(filtered);
}
renderTable(data3);
</script>
</body></html>"""

with open("dashboard_final.html","w",encoding="utf-8") as f:
    f.write(html)
print("dashboard_final.html saved!")


dashboard_final.html saved!
